# Exp 17 — Обзор русских датасетов

Задача: посмотреть на каждый из 6 кандидатов (см. `datasets.md`), понять
структуру, содержимое JSON-колонок и потенциал для entity resolution.

Feature engineering — **после** согласования с пользователем по каждому датасету.

Датасеты:
1. `kirill57678/lamoda-products` — товары Lamoda, JSON `About`
3. `egorledyaev/russian-car-market-dataset` — объявления авто
4. `fiftin/ozon-what-products-do-users-add-to-favs` — товары Ozon
5. `rzabolotin/auto-ru-car-ads-parsed` — авто, JSON `complectation_dict`
6. `snezhanausova/all-auto-ru-14-11-2020csv` — авто, JSON `equipment_dict`
7. `ruslanusov/dataset-of-electronics-with-lifecycle-and-specs` — устройства, JSON `technical_specs` (основной для нашей задачи)

Запуск:
1. `uv sync` — поставит `kagglehub`.
2. Один раз авторизоваться в Kaggle: в ячейке ниже запустить `kagglehub.login()` — введёт username + token (https://www.kaggle.com/settings → API → Create New Token) и сохранит их в `~/.config/kagglehub/`.
3. Дальше просто прогнать ноутбук.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import kagglehub
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 240)

RAW_DIR = Path("data/raw_ru")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Если kagglehub ещё не авторизован на этой машине — раскомментировать и запустить один раз:
# kagglehub.login()

In [2]:
def summarize(df: pd.DataFrame, name: str) -> None:
    """Сводка: размер, dtypes, доля пустых, примеры уникальных."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print()
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True),
        "n_null": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
    })
    print(info.to_string())


def sniff_json_column(df: pd.DataFrame, col: str, n_samples: int = 50) -> Counter:
    """Посчитать, какие ключи чаще всего встречаются в JSON-колонке."""
    keys: Counter = Counter()
    for val in df[col].dropna().head(n_samples * 20):
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = json.loads(val)
            elif isinstance(val, dict):
                parsed = val
            else:
                continue
            if isinstance(parsed, dict):
                keys.update(parsed.keys())
        except (json.JSONDecodeError, TypeError):
            continue
    return keys


def duplicate_signals(df: pd.DataFrame, subset_cols: list[str], name: str) -> None:
    """Какой процент строк — потенциальные дубли по заданному набору ключей."""
    sub = df[subset_cols].dropna(how="any")
    grouped = sub.groupby(subset_cols).size()
    n_groups = len(grouped)
    n_dup_groups = int((grouped > 1).sum())
    n_dup_rows = int(grouped[grouped > 1].sum())
    print(f"[{name}] по {subset_cols}: "
          f"{n_dup_groups}/{n_groups} групп с >1 строкой "
          f"(≈{n_dup_rows} строк, {100 * n_dup_rows / max(len(sub), 1):.1f}% от непустых)")


def _looks_structured(s: str) -> str:
    """Эвристика: похоже ли на JSON/list/dict/url/html."""
    st = s.lstrip()
    if st.startswith(("{", "[")):
        return "JSON-like"
    if st.startswith(("http://", "https://", "//")):
        return "URL"
    if st.startswith("<") and ">" in st[:50]:
        return "HTML-like"
    return ""


def show_samples(
    df: pd.DataFrame, name: str, max_len: int = 300, n: int = 2,
) -> None:
    """Для каждой object-колонки показать n непустых значений + флаг структуры.

    Цель — поймать JSON-колонки, URL-колонки, списки в строках и т.п.,
    которые не видно из одного dtype.
    """
    print(f"=== samples: {name} ===\n")
    for col in df.columns:
        dt = str(df[col].dtype)
        if dt != "object":
            print(f"· {col}  [{dt}]  — skip")
            continue
        samples = df[col].dropna().astype(str).head(n).tolist()
        if not samples:
            print(f"· {col}  [object]  — all null")
            continue
        flags = {_looks_structured(s) for s in samples if s}
        flags.discard("")
        tag = f"  ({', '.join(sorted(flags))})" if flags else ""
        print(f"· {col}{tag}")
        for s in samples:
            preview = s if len(s) <= max_len else s[:max_len] + " …"
            print(f"    {preview}")
        print()

## 1. Lamoda products

В колонке `About` лежит dict со свойствами товара (`Состав`, `Сезон`, `Размер`…).
Для ER нужны только атрибуты товара, фото — выкидываем.

In [3]:
lamoda_path = Path(kagglehub.dataset_download("kirill57678/lamoda-products"))
print("→", lamoda_path)
for p in sorted(lamoda_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(lamoda_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/kirill57678/lamoda-products/versions/1
  products.csv  (49.7 MB)


In [4]:
lamoda_csvs = sorted(lamoda_path.rglob("*.csv"))
lamoda_csvs

[PosixPath('/home/user/.cache/kagglehub/datasets/kirill57678/lamoda-products/versions/1/products.csv')]

In [5]:
lamoda = pd.read_csv(lamoda_csvs[0])
summarize(lamoda, "Lamoda")

=== Lamoda ===
shape: (10338, 8)
memory: 55.4 MB

          dtype  n_unique  n_null  null_%
Id       object     10338       0     0.0
Title    object      1979       0     0.0
Brand    object      1093       0     0.0
About    object     10319       0     0.0
Price    object      2707    3373    32.6
Image    object     10337       0     0.0
Alsobuy  object     10258       0     0.0
Similar  object     10336       0     0.0


In [6]:
show_samples(lamoda, "Lamoda")

=== samples: Lamoda ===

· Id
    mp002xm0bbeg
    mp002xm0bbel

· Title
    Брюки спортивные
    Брюки спортивные

· Brand
    O'STIN
    O'STIN

· About  (JSON-like)
    {'Состав, %': 'Хлопок - 60%, Полиэстер - 40%', 'Сезон': 'мульти', 'Размер товара на модели': 'M INT', 'Длина по внутреннему шву (см)': '71 см', 'Длина по боковому шву': '103 см', 'Ширина по низу': '17 см', 'Высота': '35 см', 'Параметры модели': '100 - 84 -94', 'Артикул': 'MP002XM0BBEG'}
    {'Состав, %': 'Хлопок - 60%, Полиэстер - 40%', 'Сезон': 'мульти', 'Размер товара на модели': 'M INT', 'Длина по внутреннему шву (см)': '71 см', 'Длина по боковому шву': '100 см', 'Высота': '31 см', 'Параметры модели': '103 - 78 - 98', 'Рост модели на фото': '189 см', 'Артикул': 'MP002XM0BBEL'}

· Price
    680 ₽
    680 ₽

· Image  (URL)
    //a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg
    //a.lmcdn.ru/img600x866/M/P/MP002XM0BBEL_24107566_1_v1_2x.jpg

· Alsobuy  (JSON-like)
    ['https://a.lmcdn.ru/img236x341/M/P/

In [7]:
lamoda.head(3)

,Id,Title,Brand,About,Price,Image,Alsobuy,Similar
0,mp002xm0bbeg,Брюки спортивные,O'STIN,"{'Состав, %': 'Хлопок - 60%, Полиэстер - 40%', 'Сезон': 'мульти', 'Размер товара на модели': 'M INT', 'Длина по внутреннему шву (см)': '71 см', 'Длина по боковому шву': '103 см', 'Ширина по низу':...",680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg,"['https://a.lmcdn.ru/img236x341/M/P/MP002XM0BOLJ_24817791_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BH7L_24718386_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BOFG_24883...","['https://a.lmcdn.ru/img236x341/M/P/MP002XM0B6E3_23944805_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0B6EC_23944813_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BH8N_25281..."
1,mp002xm0bbel,Брюки спортивные,O'STIN,"{'Состав, %': 'Хлопок - 60%, Полиэстер - 40%', 'Сезон': 'мульти', 'Размер товара на модели': 'M INT', 'Длина по внутреннему шву (см)': '71 см', 'Длина по боковому шву': '100 см', 'Высота': '31 см'...",680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEL_24107566_1_v1_2x.jpg,"['https://a.lmcdn.ru/img236x341/M/P/MP002XM0BBF0_24097007_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BBFE_24251065_1_v1.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM08RSS_17840892...","['https://a.lmcdn.ru/img236x341/M/P/MP002XM046O2_20517394_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BBEJ_24252962_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM0BC2A_24591..."
2,mp002xm0c0a7,Термобелье Optima,Polar Wolf,"{'Состав, %': 'Полиэстер - 95%, Вискоза - 5%', 'Материал подкладки, %': 'без подкладки', 'Сезон': 'мульти', 'Размер товара на модели': '50/176', 'Параметры модели': '100-89-103', 'Рост модели на ф...",1 941 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0C0A7_25186820_1_v1.jpeg,"['https://a.lmcdn.ru/img236x341/M/P/MP002XM08UEP_18074322_1_v1.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XU0566R_17530097_1_v6.jpg', 'https://a.lmcdn.ru/img236x341/R/T/RTLADU164501_24697394_1_...","['https://a.lmcdn.ru/img236x341/M/P/MP002XM0BW5V_24977362_1_v1.jpeg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM095UF_20952850_1_v1_2x.jpg', 'https://a.lmcdn.ru/img236x341/M/P/MP002XM08WNF_1824520..."


In [8]:
# Разбор JSON-колонки About
about_col = next((c for c in lamoda.columns if c.lower() == "about"), None)
print("About column:", about_col)
if about_col:
    print("\nПример сырого значения:")
    print(lamoda[about_col].dropna().iloc[0])
    print("\nЧастоты ключей (top 30):")
    for k, n in sniff_json_column(lamoda, about_col).most_common(30):
        print(f"  {n:5d}  {k}")

About column: About

Пример сырого значения:
{'Состав, %': 'Хлопок - 60%, Полиэстер - 40%', 'Сезон': 'мульти', 'Размер товара на модели': 'M INT', 'Длина по внутреннему шву (см)': '71 см', 'Длина по боковому шву': '103 см', 'Ширина по низу': '17 см', 'Высота': '35 см', 'Параметры модели': '100 - 84 -94', 'Артикул': 'MP002XM0BBEG'}

Частоты ключей (top 30):
    997  Артикул
    994  Состав, %
    993  Сезон
    782  Размер товара на модели
    476  Параметры модели
    474  Длина (см)
    460  Материал подкладки, %
    401  Цвет
    386  Длина рукава (см)
    336  Утеплитель
    308  Рост модели на фото
    301  Узор
    238  Флисовая подкладка
    195  Длина по внутреннему шву (см)
    183  Страна производства
    181  Длина по боковому шву
    145  Вид спорта
    139  Ширина по низу
    112  Подкладка
    107  Застежка
     97  Вырез/воротник
     94  Высота
     93  Сезон 2
     92  Гарантийный срок
     33  Фасон
     24  Экосостав
     21  Комплектация
     21  Recycle
     20  Сох

In [9]:
# Потенциальные дубликаты — в Lamoda колонка Title (не Name)
for subset in (["Title"], ["Title", "Brand"], ["Title", "Brand", "Price"]):
    cols = [c for c in subset if c in lamoda.columns]
    if len(cols) == len(subset):
        duplicate_signals(lamoda, cols, "Lamoda")

[Lamoda] по ['Title']: 490/1979 групп с >1 строкой (≈8849 строк, 85.6% от непустых)
[Lamoda] по ['Title', 'Brand']: 1698/5351 групп с >1 строкой (≈6685 строк, 64.7% от непустых)
[Lamoda] по ['Title', 'Brand', 'Price']: 631/6078 групп с >1 строкой (≈1518 строк, 21.8% от непустых)


## 3. Russian car market (egorledyaev)

In [10]:
cars_ru_path = Path(kagglehub.dataset_download("egorledyaev/russian-car-market-dataset"))
print("→", cars_ru_path)
for p in sorted(cars_ru_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(cars_ru_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/egorledyaev/russian-car-market-dataset/versions/1
  sample_submission.csv  (0.6 MB)
  test.csv  (25.9 MB)
  train.csv  (103.9 MB)


In [11]:
cars_ru_csvs = sorted(cars_ru_path.rglob("*.csv"))
cars_ru_csvs

[PosixPath('/home/user/.cache/kagglehub/datasets/egorledyaev/russian-car-market-dataset/versions/1/sample_submission.csv'),
 PosixPath('/home/user/.cache/kagglehub/datasets/egorledyaev/russian-car-market-dataset/versions/1/test.csv'),
 PosixPath('/home/user/.cache/kagglehub/datasets/egorledyaev/russian-car-market-dataset/versions/1/train.csv')]

In [12]:
# В датасете test.csv + возможно train.csv — читаем всё
cars_ru = {p.stem: pd.read_csv(p) for p in cars_ru_csvs}
for name, df in cars_ru.items():
    summarize(df, f"cars_ru/{name}")
    print()

=== cars_ru/sample_submission ===
shape: (19991, 2)
memory: 0.3 MB

             dtype  n_unique  n_null  null_%
id           int64     19991       0     0.0
price_rub  float64     19991       0     0.0

=== cars_ru/test ===
shape: (19991, 26)
memory: 53.1 MB

                 dtype  n_unique  n_null  null_%
id               int64     19991       0     0.0
mark            object       134       0     0.0
model           object      1233       0     0.0
generation      object       765    1993    10.0
configuration   object       173       0     0.0
complectation   object      1373   13666    68.4
body_type       object        24       0     0.0
color           object        16       1     0.0
displacement   float64        65       0     0.0
drive_type      object         3       0     0.0
engine_type     object         5       0     0.0
horse_power      int64       387       0     0.0
transmission    object         4       0     0.0
wheel           object         2       0     0.0
km_a

In [13]:
# Берём test/train (в sample_submission только id + price_rub)
cars_main = cars_ru.get("train") if "train" in cars_ru else cars_ru.get("test")
print(f"Используем: {'train' if 'train' in cars_ru else 'test'}, shape={cars_main.shape}")
cars_main.head(3)

Используем: train, shape=(80009, 27)


,id,price_rub,mark,model,generation,configuration,complectation,body_type,color,displacement,drive_type,engine_type,horse_power,transmission,wheel,km_age,year,owners_count,condition,custom,pts,seller_type,region,city,address,options,description
0,16799418,2200000,Kia,Sportage,IV Рестайлинг,Внедорожник 5 дв.,Prestige Black Edition,пятидверный внедорожник,чёрный,2.4,полный,бензин,184,автомат,левый,51150.0,2020,1,среднее,растаможен,оригинал,частник,Вологодская область,Череповец,"Россия, Вологодская область,","[""airbag-curtain"",""park-assist-r"",""leather"",""bluetooth"",""lock"",""wheel-leather"",""leather-gear-stick"",""tyre-pressure"",""drl"",""dha"",""auto-mirrors"",""esp"",""driver-seat-updown"",""airbag-driver"",""airbag-pa...","Автомобиль в отличном состоянии, вложений никаких не требует. 2,4 атмосферный двигатель на обычном автомате Полный привод! Куплен не в кредит. Своевременно обслуживался. Прошёл все ТО. Есть сервис..."
1,14261299,300000,Ford,Maverick,III,Внедорожник 5 дв.,NaN,пятидверный внедорожник,серебристый,3.0,полный,бензин,197,автомат,левый,250000.0,2001,4,среднее,растаможен,дубликат,частник,Краснодарский край,Сочи,Теневой переулок,NaN,"Есть жизненные повреждения, автомобиль рабочий, состояние соответствующее, по ходовой и технике все хорошо, езжу каждый день. руки естественно приложить есть куда"
2,10850488,2600000,Volkswagen,Caravelle,T6,Минивэн,Trendline,минивэн,чёрный,2.0,передний,дизель,102,механика,левый,360000.0,2015,2,среднее,растаможен,оригинал,частник,Оренбургская область,Орск,NaN,"[""isofix"",""front-seats-heat"",""hcc"",""heated-wash-system"",""light-sensor"",""lock"",""mirrors-heat"",""immo"",""12v-socket"",""abs"",""airbag-driver"",""airbag-passenger"",""audiopreparation"",""aux"",""bluetooth"",""cond...","Автомобиль в отличном состоянии, все то пройдены, масла фильтра поменяны. Ходовая в идеальном состоянии, трёх зонный климат контроль в рабочем состоянии. Два комплекта резины. Торг"


In [14]:
show_samples(cars_main, "cars_ru (train/test)")

=== samples: cars_ru (train/test) ===

· id  [int64]  — skip
· price_rub  [int64]  — skip
· mark
    Kia
    Ford

· model
    Sportage
    Maverick

· generation
    IV Рестайлинг
    III

· configuration
    Внедорожник 5 дв.
    Внедорожник 5 дв.

· complectation
    Prestige Black Edition
    Trendline

· body_type
    пятидверный внедорожник
    пятидверный внедорожник

· color
    чёрный
    серебристый

· displacement  [float64]  — skip
· drive_type
    полный
    полный

· engine_type
    бензин
    бензин

· horse_power  [int64]  — skip
· transmission
    автомат
    автомат

· wheel
    левый
    левый

· km_age  [float64]  — skip
· year  [int64]  — skip
· owners_count  [int64]  — skip
· condition
    среднее
    среднее

· custom
    растаможен
    растаможен

· pts
    оригинал
    дубликат

· seller_type
    частник
    частник

· region
    Вологодская область
    Краснодарский край

· city
    Череповец
    Сочи

· address
    Россия, Вологодская область, 
    Теневой пе

In [15]:
# Дубли по реальным колонкам (mark + model, + модификаторы)
for subset in (
    ["mark", "model"],
    ["mark", "model", "year"],
    ["mark", "model", "year", "generation"],
    ["mark", "model", "year", "generation", "configuration"],
    ["mark", "model", "year", "generation", "configuration", "complectation"],
):
    cols = [c for c in subset if c in cars_main.columns]
    if len(cols) == len(subset):
        duplicate_signals(cars_main, cols, "cars_ru")

[cars_ru] по ['mark', 'model']: 1467/1926 групп с >1 строкой (≈79550 строк, 99.4% от непустых)
[cars_ru] по ['mark', 'model', 'year']: 7164/10970 групп с >1 строкой (≈76203 строк, 95.2% от непустых)
[cars_ru] по ['mark', 'model', 'year', 'generation']: 6765/10375 групп с >1 строкой (≈68530 строк, 95.0% от непустых)
[cars_ru] по ['mark', 'model', 'year', 'generation', 'configuration']: 7664/12445 групп с >1 строкой (≈67359 строк, 93.4% от непустых)
[cars_ru] по ['mark', 'model', 'year', 'generation', 'configuration', 'complectation']: 4673/10278 групп с >1 строкой (≈18328 строк, 76.6% от непустых)


## 4. Ozon favorites

In [16]:
ozon_path = Path(kagglehub.dataset_download("fiftin/ozon-what-products-do-users-add-to-favs"))
print("→", ozon_path)
for p in sorted(ozon_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(ozon_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/fiftin/ozon-what-products-do-users-add-to-favs/versions/1
  2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx  (2.5 MB)
  chto-dobavlyali-v-izbrannoe-v-dekabre-2020.xlsx  (2.5 MB)
  chto-dobavlyali-v-izbrannoe-v-noyabre-2020_2SSM2SO.xlsx  (1.7 MB)
  chto-dobavlyali-v-izbrannoe-v-yanvare-2021.xlsx  (2.5 MB)
  chto-dobavlyali-v-izbrannoe_-06_02_2021-07_03_2021.xlsx  (2.4 MB)
  chto-dobavlyali-v-izbrannoe_-13_02_2021-14_03_2021.xlsx  (2.4 MB)
  chto-dobavlyali-v-izbrannoe_-16_01_2021-14_02_2021.xlsx  (2.1 MB)
  chto-dobavlyali-v-izbrannoe_-20_02_2021-21_03_2021.xlsx  (2.4 MB)
  chto-dobavlyali-v-izbrannoe_-23_01_2021-21_02_2021.xlsx  (2.4 MB)
  chto-dobavlyali-v-izbrannoe_-27_02_2021-28_03_2021.xlsx  (2.1 MB)
  chto-dobavlyali-v-izbrannoe_-30_01_2021-28_02_2021_4I3ycAJ.xlsx  (2.4 MB)
  chto-dobavlyaut-v-izbrannoe_-01_05_2021-30_05_2021.xlsx  (2.5 MB)
  chto-dobavlyaut-v-izbrannoe_-03_04_2021-02_05_2021.xlsx  (2.1 MB)
  chto-dobavlyaut-v

In [17]:
ozon_files = sorted(
    p for p in ozon_path.rglob("*")
    if p.suffix.lower() in {".csv", ".json", ".parquet", ".xlsx", ".xls"}
)
print(f"Файлов: {len(ozon_files)}")
for f in ozon_files[:3]:
    print(" ", f.name)
if len(ozon_files) > 3:
    print(f"  ... и ещё {len(ozon_files) - 3}")

Файлов: 34
  2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx
  chto-dobavlyali-v-izbrannoe-v-dekabre-2020.xlsx
  chto-dobavlyali-v-izbrannoe-v-noyabre-2020_2SSM2SO.xlsx
  ... и ещё 31


In [18]:
def _read_any(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suf == ".json":
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)
    raise ValueError(path)

# Читаем все файлы + помечаем источником (имя файла — период)
ozon_frames = []
for f in ozon_files:
    df = _read_any(f)
    df["__source_file"] = f.name
    ozon_frames.append(df)
ozon_all = pd.concat(ozon_frames, ignore_index=True)
summarize(ozon_all, "ozon_all (concat по всем периодам)")

/tmp/ipykernel_4066141/472315088.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ozon_all = pd.concat(ozon_frames, ignore_index=True)


=== ozon_all (concat по всем периодам) ===
shape: (333738, 14)
memory: 299.8 MB

                                                          dtype  n_unique  n_null  null_%
Название товара                                          object    170160     946     0.3
Бренд                                                    object     20991   27500     8.2
Ссылка на товар                                          object    198493       0     0.0
Категория 1 уровня                                       object        25       0     0.0
Категория 2 уровня                                       object       188       0     0.0
Категория 3 уровня                                       object       915       0     0.0
Категория 4 уровня                                       object      5003       0     0.0
Количество добавлений в избранное, 30 дней              float64      1792   23749     7.1
Количество добавлений в избранное, все время     datetime64[ns]       222  323738    97.0
Последнее появление

In [19]:
show_samples(ozon_all, "ozon_all")

=== samples: ozon_all ===

· Название товара
    Дневник школьный, Кошечка, розовый
    Игровая консоль PlayStation 5 Digital Edition, белый

· Бренд
    bumbel
    PlayStation

· Ссылка на товар  (URL)
    https://www.ozon.ru/product/244927827
    https://www.ozon.ru/product/178715781

· Категория 1 уровня
    Товары для мам и детей
    ТВ и аудио

· Категория 2 уровня
    Товары для школы, канцелярия, оборудование
    Игровые приставки и аксессуары TV&Audio

· Категория 3 уровня
    Бумага офисная и бумажная продукция
    Игровая приставка TV&Audio

· Категория 4 уровня
    Школьный дневник
    Игровая приставка

· Количество добавлений в избранное, 30 дней  [float64]  — skip
· Количество добавлений в избранное, все время  [datetime64[ns]]  — skip
· Последнее появление в наличии  [datetime64[ns]]  — skip
· __source_file
    2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx
    2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx

· Количество добавлений в изб

In [20]:
ozon_all.head(3)

,Название товара,Бренд,Ссылка на товар,Категория 1 уровня,Категория 2 уровня,Категория 3 уровня,Категория 4 уровня,"Количество добавлений в избранное, 30 дней","Количество добавлений в избранное, все время",Последнее появление в наличии,__source_file,"Количество добавлений в избранное, декабрь 2020","Количество добавлений в избранное, ноябрь 2020","Количество добавлений в избранное, январь 2021"
0,"Дневник школьный, Кошечка, розовый",bumbel,https://www.ozon.ru/product/244927827,Товары для мам и детей,"Товары для школы, канцелярия, оборудование",Бумага офисная и бумажная продукция,Школьный дневник,3280.0,2021-06-21,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN
1,"Игровая консоль PlayStation 5 Digital Edition, белый",PlayStation,https://www.ozon.ru/product/178715781,ТВ и аудио,Игровые приставки и аксессуары TV&Audio,Игровая приставка TV&Audio,Игровая приставка,3059.0,2021-06-24,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN
2,Natura Siberica LAB Biome Сыворотка для лица Себорегулирующая 30 мл,Natura Siberica,https://www.ozon.ru/product/242886982,Красота и здоровье,Косметика для ухода за кожей,Лицо,Сыворотки и дополнительный уход для лица,2921.0,2021-06-17,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN


In [21]:
# В Ozon один и тот же товар (SKU) мог попасть в несколько месячных выгрузок
# — это естественный источник пар-дубликатов.
for subset in (
    ["Наименование"],
    ["Наименование", "Бренд"],
    ["Наименование", "Бренд", "Категория"],
):
    cols = [c for c in subset if c in ozon_all.columns]
    if len(cols) == len(subset):
        duplicate_signals(ozon_all, cols, "ozon_all")

print("\nКолонки:", list(ozon_all.columns))


Колонки: ['Название товара', 'Бренд', 'Ссылка на товар', 'Категория 1 уровня', 'Категория 2 уровня', 'Категория 3 уровня', 'Категория 4 уровня', 'Количество добавлений в избранное, 30 дней', 'Количество добавлений в избранное, все время', 'Последнее появление в наличии', '__source_file', 'Количество добавлений в избранное, декабрь 2020', 'Количество добавлений в избранное, ноябрь 2020', 'Количество добавлений в избранное, январь 2021']


## 5. auto.ru parsed (rzabolotin)

JSON-колонка `complectation_dict` — комплектации авто (ключ→bool/str/int).

In [22]:
auto_ru_path = Path(kagglehub.dataset_download("rzabolotin/auto-ru-car-ads-parsed"))
print("→", auto_ru_path)
for p in sorted(auto_ru_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(auto_ru_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/rzabolotin/auto-ru-car-ads-parsed/versions/1
  data1.csv  (332.0 MB)


In [23]:
auto_ru_files = sorted(p for p in auto_ru_path.rglob("*") if p.suffix.lower() in {".csv", ".json", ".parquet"})
auto_ru_files

[PosixPath('/home/user/.cache/kagglehub/datasets/rzabolotin/auto-ru-car-ads-parsed/versions/1/data1.csv')]

In [24]:
auto_ru = _read_any(auto_ru_files[0])
summarize(auto_ru, "auto_ru")

/tmp/ipykernel_4066141/472315088.py:4: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


=== auto_ru ===
shape: (70896, 34)
memory: 552.4 MB

                        dtype  n_unique  n_null  null_%
bodyType               object        25       0     0.0
brand                  object        12       0     0.0
car_url                object     60424       0     0.0
color                  object        16       0     0.0
complectation_dict     object      5071   28268    39.9
description            object     52022    1000     1.4
engineDisplacement    float64        56      55     0.1
enginePower           float64       328       0     0.0
equipment_dict         object     46974    9996    14.1
fuelType               object         5       0     0.0
image                  object     70536       0     0.0
mileage                 int64     15830       0     0.0
modelDate             float64        68       0     0.0
model_info             object      1880       0     0.0
model_name             object       733       0     0.0
name                   object      3104       0    

In [25]:
show_samples(auto_ru, "auto_ru")

=== samples: auto_ru ===

· bodyType
    лифтбек
    лифтбек

· brand
    SKODA
    SKODA

· car_url  (URL)
    https://auto.ru/cars/used/sale/skoda/octavia/1100575026-c780dc09/
    https://auto.ru/cars/used/sale/skoda/octavia/1100549428-595cadf2/

· color
    синий
    чёрный

· complectation_dict  (JSON-like)
    {"id":"20026336","name":"Ambition","available_options":["heated-wash-system","airbag-passenger","lock","door-sill-panel","electro-mirrors","mirrors-heat","cooling-box","computer","seat-transformation","wheel-power","fabric-seats","airbag-side","abs","wheel-leather","climate-control-1","esp","adaptiv …
    {"id":"20803582","name":"Ambition","available_options":["heated-wash-system","airbag-passenger","lock","electro-mirrors","mirrors-heat","computer","seat-transformation","wheel-power","fabric-seats","steel-wheels","airbag-side","abs","esp","usb","audiopreparation","ashtray-and-cigarette-lighter","ele …

· description
    Все автомобили, представленные в продаже, проходят тща

In [26]:
auto_ru.head(3)

,bodyType,brand,car_url,color,complectation_dict,description,engineDisplacement,enginePower,equipment_dict,fuelType,image,mileage,modelDate,model_info,model_name,name,numberOfDoors,parsing_unixtime,priceCurrency,productionDate,sell_id,super_gen,vehicleConfiguration,vehicleTransmission,vendor,Владельцы,Владение,ПТС,Привод,Руль,Состояние,Таможня,is_train,price
0,лифтбек,SKODA,https://auto.ru/cars/used/sale/skoda/octavia/1100575026-c780dc09/,синий,NaN,"Все автомобили, представленные в продаже, проходят тщательную проверку по более 40 параметрам. Предоставляем гарантию юридической чистоты, а так же год технической гарантии на двигатель и КПП. Бес...",1.2,105.0,"{""engine-proof"":true,""tinted-glass"":true,""airbag-driver"":true,""aux"":true,""isofix"":true,""electro-window-front"":true,""ashtray-and-cigarette-lighter"":true,""airbag-passenger"":true,""computer"":true,""hig...",бензин,https://autoru.naydex.net/o9DBXQ270/5ac010hAY0/Xkcrbmf2u0IghxJHqVi5dGL7OcugpPbM0sYLDhB9YWw7CxRKU17ysuJYxu9oaUHn7ahNSrqiKwm-CQDyDolDeEoEc3J49fgWYNYBUbQC7D96sj6K9_O-mo6XT34oWVQDBTEybGZikaX4X4bwLyUuj...,74000,2013.0,"{""code"":""OCTAVIA"",""name"":""Octavia"",""ru_name"":""Октавия"",""morphology"":{""gender"":""FEMININE""},""nameplate"":{""code"":"""",""name"":"""",""semantic_url"":""""}}",OCTAVIA,1.2 AMT (105 л.с.),5.0,1603226273,RUB,2014,1100575026,"{""id"":""10373605"",""displacement"":1197,""engine_type"":""GASOLINE"",""gear_type"":""FORWARD_CONTROL"",""transmission"":""ROBOT"",""power"":105,""power_kvt"":77,""human_name"":""1.2 AMT (105 л.с.)"",""acceleration"":10.5,...",LIFTBACK ROBOT 1.2,роботизированная,EUROPEAN,3 или более,NaN,Оригинал,передний,Левый,Не требует ремонта,Растаможен,False,0
1,лифтбек,SKODA,https://auto.ru/cars/used/sale/skoda/octavia/1100549428-595cadf2/,чёрный,NaN,ЛОТ: 01217195\nАвтопрага Север\nДанный автомобиль прошел диагностику по 147 пунктам и имеет сертификат технической гарантии.\n\n\nВы можете получить скидку на данный автомобиль до 50000 рублей. По...,1.6,110.0,"{""cruise-control"":true,""asr"":true,""esp"":true,""airbag-driver"":true,""isofix"":true,""usb"":true,""light-sensor"":true,""airbag-passenger"":true,""computer"":true,""wheel-power"":true,""alarm"":true,""lock"":true,""...",бензин,https://autoru.naydex.net/o9DBXQ270/5ac010hAY0/Xkcrbmf2u0IghxJHqVi5dGL7OcugpPbM0sYLDhB9YWw7CxRKU17ysuJYxu9kY0fj5q5NROmsLgu7CQ34WY0UI0pVdyN6pahANodRBrYA5TJ-4z2K9_O-mo6XT34oWVQDBTEybGZikaX4X4bwLyUuj...,60563,2017.0,"{""code"":""OCTAVIA"",""name"":""Octavia"",""ru_name"":""Октавия"",""morphology"":{""gender"":""FEMININE""},""nameplate"":{""code"":"""",""name"":"""",""semantic_url"":""""}}",OCTAVIA,1.6 MT (110 л.с.),5.0,1603226277,RUB,2017,1100549428,"{""id"":""20913311"",""displacement"":1598,""engine_type"":""GASOLINE"",""gear_type"":""FORWARD_CONTROL"",""transmission"":""MECHANICAL"",""power"":110,""power_kvt"":81,""human_name"":""1.6 MT (110 л.с.)"",""acceleration"":1...",LIFTBACK MECHANICAL 1.6,механическая,EUROPEAN,1 владелец,NaN,Оригинал,передний,Левый,Не требует ремонта,Растаможен,False,0
2,лифтбек,SKODA,https://auto.ru/cars/used/sale/skoda/superb/1100658222-7ac3def5/,серый,"{""id"":""20026336"",""name"":""Ambition"",""available_options"":[""heated-wash-system"",""airbag-passenger"",""lock"",""door-sill-panel"",""electro-mirrors"",""mirrors-heat"",""cooling-box"",""computer"",""seat-transformat...","Все автомобили, представленные в продаже, проходят тщательную проверку по более 40 параметрам. Предоставляем гарантию юридической чистоты, а так же год технической гарантии на двигатель и КПП. Бес...",1.8,152.0,"{""cruise-control"":true,""tinted-glass"":true,""esp"":true,""adaptive-light"":true,""multi-wheel"":true,""xenon"":true,""heated-wash-system"":true,""ashtray-and-cigarette-lighter"":true,""airbag-passenger"":true,""...",бензин,https://avatars.mds.yandex.net/get-autoru-vos/2028593/7e0dceb610c52cdb6dd771b29b0b3f0d/320x240,88000,2013.0,"{""code"":""SUPERB"",""name"":""Superb"",""ru_name"":""Суперб"",""morphology"":{},""nameplate"":

In [27]:
if "complectation_dict" in auto_ru.columns:
    print("Пример сырого complectation_dict:\n")
    print(auto_ru["complectation_dict"].dropna().iloc[0][:800])
    print("\nТоп-30 ключей:")
    for k, n in sniff_json_column(auto_ru, "complectation_dict").most_common(30):
        print(f"  {n:5d}  {k}")

Пример сырого complectation_dict:

{"id":"20026336","name":"Ambition","available_options":["heated-wash-system","airbag-passenger","lock","door-sill-panel","electro-mirrors","mirrors-heat","cooling-box","computer","seat-transformation","wheel-power","fabric-seats","airbag-side","abs","wheel-leather","climate-control-1","esp","adaptive-light","audiopreparation","ashtray-and-cigarette-lighter","front-centre-armrest","electro-window-back","16-inch-wheels","body-mouldings","condition","airbag-driver","isofix","aux","electro-window-front","light-sensor","hcc","ptf","rain-sensor","tyre-pressure","audiosystem-cd","front-seats-heat","wheel-configuration2","wheel-configuration1","immo","12v-socket","third-rear-headrest"]}

Топ-30 ключей:
   1000  id
   1000  name
   1000  available_options
    454  vendor_colors


## 6. auto.ru 14-11-2020 (snezhanausova)

JSON-колонка `equipment_dict`.

In [28]:
auto_ru_2020_path = Path(kagglehub.dataset_download("snezhanausova/all-auto-ru-14-11-2020csv"))
print("→", auto_ru_2020_path)
for p in sorted(auto_ru_2020_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(auto_ru_2020_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/snezhanausova/all-auto-ru-14-11-2020csv/versions/1
  all_auto_ru_14_11_2020.csv  (426.1 MB)


In [29]:
auto_ru_2020_files = sorted(
    p for p in auto_ru_2020_path.rglob("*") if p.suffix.lower() in {".csv", ".json", ".parquet"}
)
auto_ru_2020 = _read_any(auto_ru_2020_files[0])
summarize(auto_ru_2020, "auto_ru_2020")

=== auto_ru_2020 ===
shape: (77449, 34)
memory: 699.6 MB

                        dtype  n_unique  n_null  null_%
bodyType               object       166       2     0.0
brand                  object        36       0     0.0
color                  object        16       0     0.0
complectation_dict     object      5226       0     0.0
description            object     61684    2667     3.4
engineDisplacement     object       103       0     0.0
enginePower           float64       386       2     0.0
equipment_dict         object     45548       0     0.0
fuelType               object         7       0     0.0
image                   int64        47       0     0.0
mileage                 int64     18645       0     0.0
modelDate             float64        73       2     0.0
model_info             object      1613       0     0.0
model_name             object      1064       0     0.0
name                   object      4333       2     0.0
numberOfDoors         float64         5       

In [30]:
show_samples(auto_ru_2020, "auto_ru_2020")

=== samples: auto_ru_2020 ===

· bodyType
    Седан
    Седан

· brand
    AUDI
    AUDI

· color
    97948F
    CACECB

· complectation_dict  (JSON-like)
    {'id': '0'}
    {'id': '0'}

· description
    Машина на полном ходу
Состояние хорошее
С документами полный порядок(кристально чистая)
Всего было 3 владельца 
По всем вопросам ПИСАТЬ в личные сообщения (не звонить, я вам скину другой номер телефона)
    Продажа от официального дилера KIA - Компания "АГАЛАТ".

Гарантированная скидка при покупке в КРЕДИТ 20.000 руб.
Выгода при ОБМЕНЕ до 50.000 руб.
АВТОМОБИЛЬ ПРОШЕЛ ПРЕДПРОДАЖНУЮ ПОДГОТОВКУ
ГАРАНТИЯ ЮРИДИЧЕСКОЙ ЧИСТОТЫ АВТОМОБИЛЯ 100%
ВОЗМОЖНОСТЬ ОБМЕНА ПРЕДСТАВЛЕННОГО АВТОМОБИЛЯ НА ВАШ

Кредит от 9, …

· engineDisplacement
    2.3
    1.8

· enginePower  [float64]  — skip
· equipment_dict  (JSON-like)
    {}
    {'condition': True, 'audiosystem-cd': True, 'fabric-seats': True}

· fuelType
    бензин
    бензин

· image  [int64]  — skip
· mileage  [int64]  — skip
· modelDate  [floa

In [31]:
auto_ru_2020.head(3)

,bodyType,brand,color,complectation_dict,description,engineDisplacement,enginePower,equipment_dict,fuelType,image,mileage,modelDate,model_info,model_name,name,numberOfDoors,start_date,priceCurrency,productionDate,sell_id,super_gen,vehicleConfiguration,vehicleTransmission,vendor,Владельцы,Владение,ПТС,Привод,Руль,Состояние,Таможня,price,price_EUR,price_USD
0,Седан,AUDI,97948F,{'id': '0'},"Машина на полном ходу\nСостояние хорошее\nС документами полный порядок(кристально чистая)\nВсего было 3 владельца \nПо всем вопросам ПИСАТЬ в личные сообщения (не звонить, я вам скину другой номер...",2.3,133.0,{},бензин,5,359000,1990.0,"{'code': '100', 'name': '100', 'ru_name': '100', 'morphology': {}, 'nameplate': {'code': '', 'name': '', 'semantic_url': ''}}",100,2.3 MT (133 л.с.),4.0,2020-11-13T23:31:09Z,RUR,1993,1101578354,"{'id': '7879487', 'displacement': 2309, 'engine_type': 'GASOLINE', 'gear_type': 'FORWARD_CONTROL', 'transmission': 'MECHANICAL', 'power': 133, 'power_kvt': 98, 'human_name': '2.3 MT (133 л.с.)', '...",SEDAN MECHANICAL 2.3,MECHANICAL,EUROPEAN,3.0,NaN,ORIGINAL,передний,LEFT,True,True,106000.0,1161.0,1371.0
1,Седан,AUDI,CACECB,{'id': '0'},"Продажа от официального дилера KIA - Компания ""АГАЛАТ"".\n\nГарантированная скидка при покупке в КРЕДИТ 20.000 руб.\nВыгода при ОБМЕНЕ до 50.000 руб.\nАВТОМОБИЛЬ ПРОШЕЛ ПРЕДПРОДАЖНУЮ ПОДГОТОВКУ\nГА...",1.8,90.0,"{'condition': True, 'audiosystem-cd': True, 'fabric-seats': True}",бензин,7,204700,1982.0,"{'code': '100', 'name': '100', 'ru_name': '100', 'morphology': {}, 'nameplate': {'code': '', 'name': '', 'semantic_url': ''}}",100,1.8 MT (90 л.с.),4.0,2020-11-12T09:13:49Z,RUR,1984,1101559676,"{'id': '20388757', 'displacement': 1781, 'engine_type': 'GASOLINE', 'gear_type': 'FORWARD_CONTROL', 'transmission': 'MECHANICAL', 'power': 90, 'power_kvt': 66, 'human_name': '1.8 MT (90 л.с.)', 'a...",SEDAN MECHANICAL 1.8,MECHANICAL,EUROPEAN,3.0,NaN,ORIGINAL,передний,LEFT,True,True,44000.0,482.0,569.0
2,Седан,AUDI,040001,{'id': '0'},Автомобиль продает Официальный дилер Hyundai. Компания Марс Моторс.\n\nВозможна покупка в кредит с использованием различных программ кредитования.\nВозможен обмен на ваш автомобиль с предоставлени...,2.6,150.0,"{'airbag-driver': True, 'isofix': True, 'electro-window-front': True, 'audiosystem-cd': True, 'driver-seat-updown': True, 'airbag-passenger': True, 'computer': True, 'abs': True, 'power-child-lock...",бензин,11,403000,1990.0,"{'code': '100', 'name': '100', 'ru_name': '100', 'morphology': {}, 'nameplate': {'code': '', 'name': '', 'semantic_url': ''}}",100,2.6 MT (150 л.с.),4.0,2020-11-12T10:14:26Z,RUR,1992,1101560438,"{'id': '7879492', 'displacement': 2598, 'engine_type': 'GASOLINE', 'gear_type': 'FORWARD_CONTROL', 'transmission': 'MECHANICAL', 'power': 150, 'power_kvt': 110, 'human_name': '2.6 MT (150 л.с.)', ...",SEDAN MECHANICAL 2.6,MECHANICAL,EUROPEAN,3.0,NaN,DUPLICATE,передний,LEFT,True,True,167000.0,1829.0,2160.0


In [32]:
if "equipment_dict" in auto_ru_2020.columns:
    # Первый непустой dict (пропускаем '{}')
    sample = None
    for v in auto_ru_2020["equipment_dict"].dropna():
        s = str(v).strip()
        if s and s not in ("{}", "''", '""'):
            sample = s
            break
    print("Пример непустого equipment_dict:\n")
    print((sample or "<все пустые>")[:800])
    print("\nТоп-30 ключей:")
    for k, n in sniff_json_column(auto_ru_2020, "equipment_dict", n_samples=200).most_common(30):
        print(f"  {n:5d}  {k}")

    # Сколько строк с пустым dict vs содержательным
    eq = auto_ru_2020["equipment_dict"].dropna().astype(str).str.strip()
    n_empty = int((eq == "{}").sum())
    print(f"\nequipment_dict == '{{}}': {n_empty}/{len(eq)} ({100*n_empty/len(eq):.1f}%)")

Пример непустого equipment_dict:

{'condition': True, 'audiosystem-cd': True, 'fabric-seats': True}

Топ-30 ключей:

equipment_dict == '{}': 17732/77449 (22.9%)


## 7. DeviceStatus 15K (основной датасет)

Используем русский вариант `device_dataset_with_status_15000_ru.json`.
Вложенный `technical_specs` нужно распарсить в отдельные колонки.

In [33]:
devices_path = Path(
    kagglehub.dataset_download(
        "ruslanusov/dataset-of-electronics-with-lifecycle-and-specs"
    )
)
print("→", devices_path)
for p in sorted(devices_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(devices_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

→ /home/user/.cache/kagglehub/datasets/ruslanusov/dataset-of-electronics-with-lifecycle-and-specs/versions/1
  device_dataset_with_status_15000_en.json  (6.7 MB)
  device_dataset_with_status_15000_ru.json  (7.1 MB)


In [34]:
ru_file = next(
    p for p in devices_path.rglob("device_dataset_with_status_15000_ru.json")
)
with open(ru_file, encoding="utf-8") as f:
    raw = json.load(f)
print(f"records: {len(raw)}")
print("\nПример первой записи:")
print(json.dumps(raw[0], ensure_ascii=False, indent=2))

records: 15000

Пример первой записи:
{
  "id": "bac766ee-bfae-462e-8fb2-01b24476c4a0",
  "device_model": "ПРО-352",
  "model_family": "ПРО-Series",
  "manufacturer": "Lenovo",
  "device_type": "Проектор",
  "release_year": 2013,
  "status": "на складе",
  "usage_intensity": "low",
  "climate_zone": "moderate",
  "technical_specs": {
    "brightness_lumens": 3000,
    "resolution": "Full HD"
  }
}


In [35]:
devices = pd.json_normalize(raw, sep=".")
summarize(devices, "devices_ru")

=== devices_ru ===
shape: (15000, 45)
memory: 20.0 MB

                                       dtype  n_unique  n_null  null_%
id                                    object     15000       0     0.0
device_model                          object      9611       0     0.0
model_family                          object        18       0     0.0
manufacturer                          object        10       0     0.0
device_type                           object        19       0     0.0
release_year                           int64        19       0     0.0
status                                object         2       0     0.0
usage_intensity                       object         3       0     0.0
climate_zone                          object         3       0     0.0
technical_specs.brightness_lumens    float64         1   14241    94.9
technical_specs.resolution            object         3   12682    84.5
technical_specs.frame_rate           float64         1   14222    94.8
start_year            

In [36]:
show_samples(devices, "devices_ru")

=== samples: devices_ru ===

· id
    bac766ee-bfae-462e-8fb2-01b24476c4a0
    3e38427f-935a-438e-8412-0cf048a90c70

· device_model
    ПРО-352
    ВЕБ-192

· model_family
    ПРО-Series
    ВЕБ-Series

· manufacturer
    Lenovo
    Logitech

· device_type
    Проектор
    Веб-камера

· release_year  [int64]  — skip
· status
    на складе
    на складе

· usage_intensity
    low
    medium

· climate_zone
    moderate
    hot

· technical_specs.brightness_lumens  [float64]  — skip
· technical_specs.resolution
    Full HD
    1080p

· technical_specs.frame_rate  [float64]  — skip
· start_year  [float64]  — skip
· service_life_years  [float64]  — skip
· predicted_break_year  [float64]  — skip
· actual_break_year  [float64]  — skip
· technical_specs.capacity_gb  [float64]  — skip
· technical_specs.usb_version
    USB 3.0
    USB 3.0

· technical_specs.print
    True
    True

· technical_specs.scan
    True
    True

· technical_specs.copy
    True
    True

· technical_specs.ports  [floa

In [37]:
devices.head(3)

,id,device_model,model_family,manufacturer,device_type,release_year,status,usage_intensity,climate_zone,technical_specs.brightness_lumens,technical_specs.resolution,technical_specs.frame_rate,start_year,service_life_years,predicted_break_year,actual_break_year,technical_specs.capacity_gb,technical_specs.usb_version,technical_specs.print,technical_specs.scan,technical_specs.copy,technical_specs.ports,technical_specs.poe_support,technical_specs.speed_gbps,technical_specs.diagonal_inch,technical_specs.panel_type,technical_specs.connection,technical_specs.microphone,technical_specs.layout,technical_specs.type,technical_specs.polar_pattern,technical_specs.dpi,technical_specs.mic_count,technical_specs.protocol,technical_specs.scan_type,technical_specs.interface,technical_specs.power_va,technical_specs.battery_runtime_min,technical_specs.wifi_standard,technical_specs.color,technical_specs.pages_per_minute,technical_specs.cpu,technical_specs.ram_gb,technical_specs.storage_gb,technical_specs.scan_speed_ppm
0,bac766ee-bfae-462e-8fb2-01b24476c4a0,ПРО-352,ПРО-Series,Lenovo,Проектор,2013,на складе,low,moderate,3000.0,Full HD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3e38427f-935a-438e-8412-0cf048a90c70,ВЕБ-192,ВЕБ-Series,Logitech,Веб-камера,2016,на складе,medium,hot,NaN,1080p,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,b7b286a0-c0f5-4edf-a643-2df680aeb73e,ФЛЕ-665,ФЛЕ-Series,HP,Флешка,2019,в эксплуатации,low,cold,NaN,NaN,NaN,2021.0,7.0,2028.0,2029.0,64.0,USB 3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# technical_specs раскрыт в plain-колонки через json_normalize (technical_specs.*)
spec_cols = [c for c in devices.columns if c.startswith("technical_specs.")]
print(f"technical_specs → {len(spec_cols)} колонок:")
for c in spec_cols:
    n_null = devices[c].isna().sum()
    print(f"  {c:45s}  null={n_null:5d} ({100*n_null/len(devices):.1f}%)  "
          f"unique={devices[c].nunique():5d}")

technical_specs → 32 колонок:
  technical_specs.brightness_lumens              null=14241 (94.9%)  unique=    1
  technical_specs.resolution                     null=12682 (84.5%)  unique=    3
  technical_specs.frame_rate                     null=14222 (94.8%)  unique=    1
  technical_specs.capacity_gb                    null=14237 (94.9%)  unique=    1
  technical_specs.usb_version                    null=14237 (94.9%)  unique=    1
  technical_specs.print                          null=14175 (94.5%)  unique=    1
  technical_specs.scan                           null=14175 (94.5%)  unique=    1
  technical_specs.copy                           null=14175 (94.5%)  unique=    1
  technical_specs.ports                          null=11864 (79.1%)  unique=    4
  technical_specs.poe_support                    null=14232 (94.9%)  unique=    1
  technical_specs.speed_gbps                     null=14195 (94.6%)  unique=    1
  technical_specs.diagonal_inch                  null=14219 (94.8%) 

In [39]:
# Дубликаты в devices
for subset in (
    ["manufacturer", "device_model"],
    ["manufacturer", "device_model", "release_year"],
    ["device_type", "manufacturer", "device_model"],
):
    cols = [c for c in subset if c in devices.columns]
    if len(cols) == len(subset):
        duplicate_signals(devices, cols, "devices_ru")

# Распределение типов
print("\nРаспределение device_type:")
print(devices["device_type"].value_counts())

[devices_ru] по ['manufacturer', 'device_model']: 696/14282 групп с >1 строкой (≈1414 строк, 9.4% от непустых)
[devices_ru] по ['manufacturer', 'device_model', 'release_year']: 39/14961 групп с >1 строкой (≈78 строк, 0.5% от непустых)
[devices_ru] по ['device_type', 'manufacturer', 'device_model']: 627/14357 групп с >1 строкой (≈1270 строк, 8.5% от непустых)

Распределение device_type:
device_type
Сканер                837
ИБП                   833
МФУ                   825
Сканер штрих-кодов    820
Серверный свитч       805
Мышь                  801
IP-телефон            797
Микрофон              794
Принтер               788
Клавиатура            786
Монитор               781
Гарнитура             781
Веб-камера            778
Коммутатор            768
Маршрутизатор         766
Флешка                763
Ноутбук               762
Проектор              759
Конференц-спикер      756
Name: count, dtype: int64


In [40]:
# Какие колонки варьируются ВНУТРИ групп (manufacturer, device_model)?
# Если только id — пары ER бесполезны. Если status/climate_zone/usage_intensity — норм.
dup_groups = devices.groupby(["manufacturer", "device_model"]).filter(lambda g: len(g) > 1)
print(f"Строк в дублирующихся группах: {len(dup_groups)}")

varying = {}
for col in devices.columns:
    n_varying_groups = (
        dup_groups.groupby(["manufacturer", "device_model"])[col]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )
    if n_varying_groups > 0:
        varying[col] = int(n_varying_groups)

print("\nКолонки, которые различаются внутри хотя бы одной пары (кол-во таких групп):")
for col, n in sorted(varying.items(), key=lambda x: -x[1]):
    print(f"  {n:4d}  {col}")

# Показать пару-пример
example = dup_groups.groupby(["manufacturer", "device_model"]).head(2).head(4)
print("\nПример пары:")
cols_show = ["manufacturer", "device_model", "device_type", "release_year",
             "status", "usage_intensity", "climate_zone",
             "start_year", "service_life_years", "actual_break_year"]
print(example[[c for c in cols_show if c in example.columns]].to_string())

Строк в дублирующихся группах: 1414

Колонки, которые различаются внутри хотя бы одной пары (кол-во таких групп):
   696  id
   660  release_year
   513  actual_break_year
   511  predicted_break_year
   510  start_year
   498  service_life_years
   485  usage_intensity
   462  climate_zone
   333  status
    75  device_type
    75  technical_specs.dpi
    75  technical_specs.scan_type
    75  technical_specs.interface
    75  technical_specs.scan_speed_ppm

Пример пары:
   manufacturer device_model    device_type  release_year          status usage_intensity climate_zone  start_year  service_life_years  actual_break_year
43      TP-Link      ФЛЕ-688         Флешка          2016  в эксплуатации             low     moderate      2017.0                 8.0             2024.0
77        Cisco      МАР-776  Маршрутизатор          2006       на складе          medium         cold         NaN                 NaN                NaN
81         Dell      ВЕБ-509     Веб-камера          2019  в э

## Итог

После просмотра каждого датасета нужно согласовать с пользователем:
- Какие колонки оставить для ER, какие выкинуть (id, фото, даты скрейпа, служебные).
- Какие новые признаки посчитать (feature engineering) — решаем по содержимому.
- Как получать пары-кандидаты (дубли внутри датасета / между датасетами / порча).

Дальше: отдельные `0X_prepare_*.py` скрипты под каждый датасет.